# Chavruta.AI — הטמעה היברידית מלאה (Kaggle GPU)

מטמיע את **כל התנ"ך + 12 מפרשים (~127K צ'אנקים)** עם **bge-m3** בשני ערוצים:
- **dense** (1024-dim) — דמיון סמנטי
- **sparse** (מילוני, token→weight) — התאמה מילולית מדויקת ("רש"י", הפניות)

יחד הם מפעילים **אחזור היברידי (RRF)** במערכת — research decision D5.

⚙️ **הגדרות נדרשות (צד ימין):** Accelerator = **GPU T4 x2 / P100** · **Internet = On**

⏱️ **זמן צפוי:** ~50-70 דקות סה"כ על T4 (הורדות ~5 דק' + הטמעה ~40-60 דק' + שמירה ~5 דק').


### 1. בדוק GPU


In [ ]:
!nvidia-smi


### 2. התקנות


In [ ]:
!pip install -q -U FlagEmbedding huggingface_hub numpy


### 3. הורד את הקורפוס מ-HF (דורש Internet=On)


In [ ]:
import os, shutil
from huggingface_hub import hf_hub_download

HF_DATASET = "Yehuda-Rubin/chavruta-torah-mixed"
p = hf_hub_download(repo_id=HF_DATASET, filename="all_chunks_full.json",
                    repo_type="dataset", local_dir="/kaggle/working")
print("downloaded:", p)


### 4. טען צ'אנקים


In [ ]:
import json
from pathlib import Path

raw = json.loads(Path("/kaggle/working/all_chunks_full.json").read_text(encoding="utf-8"))
chunks = raw["chunks"] if isinstance(raw, dict) else raw
docs  = [c["document"] for c in chunks]
ids   = [c["id"] for c in chunks]
metas = [c["metadata"] for c in chunks]
print(f"chunks: {len(chunks):,}")


### 5. טען bge-m3 (FlagEmbedding — dense+sparse)


In [ ]:
from FlagEmbedding import BGEM3FlagModel

model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True, device="cuda")
print("model ready")


### 6. הטמעה — dense + sparse, עם checkpoint כל 10K

אם ה-session נופל באמצע — הרץ שוב את התא; הוא ממשיך מה-checkpoint האחרון.


In [ ]:
import numpy as np, time, pickle

BATCH, MAX_LEN, CKPT_EVERY = 128, 512, 10_000
ckpt = Path("/kaggle/working/embed_ckpt.pkl")

if ckpt.exists():
    state = pickle.loads(ckpt.read_bytes())
    dense_parts, sparse_rows, start = state["dense"], state["sparse"], state["done"]
    print(f"resuming from {start:,}")
else:
    dense_parts, sparse_rows, start = [], [], 0

t0 = time.time()
for s in range(start, len(docs), BATCH):
    batch = docs[s:s+BATCH]
    enc = model.encode(batch, batch_size=BATCH, max_length=MAX_LEN,
                       return_dense=True, return_sparse=True, return_colbert_vecs=False)
    dense_parts.append(np.asarray(enc["dense_vecs"], dtype="float32"))
    for w in enc["lexical_weights"]:
        sparse_rows.append({int(t): float(v) for t, v in dict(w).items()})
    done = min(s + BATCH, len(docs))
    if done % CKPT_EVERY < BATCH or done == len(docs):
        ckpt.write_bytes(pickle.dumps({"dense": dense_parts, "sparse": sparse_rows, "done": done}))
    rate = done / max(time.time() - t0, 1e-9)
    eta_min = (len(docs) - done) / max(rate, 1e-9) / 60
    print(f"  {done:,}/{len(docs):,}  ({rate:.0f}/s, ETA {eta_min:.0f} min)", end="\r")

vecs = np.vstack(dense_parts)
norms = np.linalg.norm(vecs, axis=1, keepdims=True); norms[norms == 0] = 1.0
vecs = vecs / norms
print(f"\ndone: dense {vecs.shape} + sparse x{len(sparse_rows):,} in {(time.time()-t0)/60:.1f} min")


### 7. שמור את שלושת קבצי הפלט


In [ ]:
out = Path("/kaggle/working/out"); out.mkdir(exist_ok=True)

np.save(out / "corpus_vectors.npy", vecs)
with open(out / "corpus_sparse.jsonl", "w", encoding="utf-8") as f:
    for i, row in enumerate(sparse_rows):
        f.write(json.dumps({"i": i, "sparse": row}) + "\n")
with open(out / "corpus_meta.jsonl", "w", encoding="utf-8") as f:
    for i, (cid, doc, meta) in enumerate(zip(ids, docs, metas)):
        f.write(json.dumps({"i": i, "id": cid, "document": doc, "metadata": meta},
                           ensure_ascii=False) + "\n")

for p in sorted(out.iterdir()):
    print(f"{p.name:24s} {p.stat().st_size/1e6:8.1f} MB")


### 8. דחוס להורדה


In [ ]:
shutil.make_archive("/kaggle/working/chavruta_hybrid_vectors", "zip", out)
print("download: /kaggle/working/chavruta_hybrid_vectors.zip")


### 9. בחזרה במחשב שלך

```powershell
# חלץ את ה-zip לתוך out\ בתיקיית הפרויקט, ואז:
python scripts/load_to_store.py --in out --profile local
python scripts/run_eval.py --retrieval-only        # השוואת היברידי מול ה-baseline
```

ההיברידי נדלק אוטומטית — הטוען מזהה את `corpus_sparse.jsonl` לבד.
